Pipeline Simulation

System Imports

In [6]:
import oqs
import time
import os
import matplotlib.pyplot as plt
import numpy as np
import random
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
import struct
from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305
import hashlib
from cryptography.hazmat.primitives.ciphers.aead import AESGCM


Demo Parameters

In [7]:
kyber_alg = "Kyber768"
dilithium_alg = "Dilithium3"
n_records = 500 # High-traffic scenario
num_partitions = 10
delta = 0.001
num_nodes = 4
enclaves_per_node = 4
device_secret = {
    "Device_1": os.urandom(32)
}
policy = {
    "role": ["doctor", "researcher"],
    "region": ["EU", "US"]
}

# Performance monitoring dictionary
performance_metrics = {
    "phase_2": {},
    "phase_3": {},
    "phase_4": {},
    "phase_6": {}
}

Phase 1

In [8]:
# HDFS generate key pair
kyber_init_start = time.perf_counter()
kem = oqs.KeyEncapsulation(kyber_alg)
hdfs_PKs = kem.generate_keypair()
hdfs_SKs = kem.export_secret_key()
print("HDFS PK:", hdfs_PKs)
print("HDFS SK:", hdfs_SKs)

#key encapsulation
hdfs_ciphertext, hdfs_shared_secret_enc = kem.encap_secret(hdfs_PKs)
kyber_init_end = time.perf_counter()
performance_metrics["phase_2"]["HDFS"] = kyber_init_end - kyber_init_start
print("HDFS Ciphertext:", hdfs_ciphertext)
print("HDFS Shared Secret (Encapsulated):", hdfs_shared_secret_enc)


HDFS PK: b'6\xf5l(2\x9b\x84v\x9cUX.\xd5t\x1d\xbegbu\xab\x9e\x1c\xcc\'\xf4t(h\x06\xd0pL8\x01\xf7\x0cq\xa3\x88\x98\xd8ob9d\xbfu\x10]\x967\xf4\x8a~\x9a\xc8\x16\x8d\x8ci\xc9\\T\xcb\xfc#7\xec\x9a`9\xcd\x04{Q\xf5\x00\x13\xfbU\xcc\xc0\xa4@\t\x88\x83\x0ck\x06\xd7{\x12\xfa\xf9\x95\xae\xa6x\xaejhk\xca\x00\xb2|)\xa8\xfc-i"_\xdd\xb3\x96)\xd2\xa6\x15\xcc;\xe8\xf3\x03\xdb\xe6 \xf6Tk\xda G\xcb:\x15=\xd8m<\x08>\xcfpp\xaeXSxs@\x16\x91\x8c$\xc2\x00!\xf0\xb0\x8f[7\x1b\xaa\x9eG9\xc8\x8f\xc5b\xc4\x10\x95\x1cd\x13[L\x83\xfa\xa1\x16 \xb4\x96]<\x93\x04y|\xb5jWG\xfa\x97\xae\x88G\x89r[\x7f\x19\x0b\x98\x81\x0c\x0b\xa5\x1b\x13\xc7&\x13\x89\xc9\xf6\xfc+\xbb\xb3\xa9\x18w\xba\x14\x15:k\xda\xc9lzT|\x872K\xa8\'\xf2\x16%m\xaa&\x18\xac\x88\x12K\x00\xb2\xf4\x15\x0e\xe3_[\xf8\x98\xb3\x98\x9f\x13KM\xcb&\xb0j\xe3\x91&\xc8\x1fC\x85\x07\x86\xd7\xb3\xa9+\x9c\x997\x7f\xd8,\xcd|\x1b\x8a\xd8`abyu9\xf7?\x8d\x0c\xb6\xab\xc85\xa1@\x10\x8eje\x85D{T\\\x93\x9d\xe1\x01\xbb\x00cP%\x19" \x8aa\x88\x1b>\xe6>\x95\xc7-\x02\x9b\x875\xe5U\xc75\

Phase 2

In [9]:
#Data acquisition
# Simulate data acquisition by generating random data

class IoTDevice:
    def __init__(self, device_id, SS):
        self.device_id = device_id
        self.counter = 0
        self.SS = SS
        self.ds = device_secret.get(device_id)  # Simulate device-specific secret for PUF
        self.ce = "epoch1"  # Simulate current epoch for authentication
        self.last_message = None  # Store the last message for potential retries

    def generate_payload(self, topic):
        if topic == "temperature":
            return round(random.uniform(20.0, 35.0), 2)
        elif topic == "humidity":
            return round(random.uniform(30.0, 80.0), 2)
        elif topic == "heart_rate":
            return random.randint(60, 120)
        else:
            return random.randint(0, 100)
    
    def encrypt_record(self, Ki, nounce, mi, aad):
        aead = ChaCha20Poly1305(Ki)
        plaintext = str(mi).encode()
        ciphertext = aead.encrypt(nounce, plaintext, aad)
        return ciphertext
    
    def simulate_puf(self, device_secret, ce):
        data = device_secret + ce.encode()
        return hashlib.sha256(data).digest()
    
    def hash_ciphertext(self, Ci):
        return hashlib.sha256(Ci).digest()

    def generate_auth(self, Ri, ce, ri_e, Ci):
        data = (
            Ri['ID'].encode() + 
            struct.pack(">d", Ri['ts']) + 
            Ri['ctr'].to_bytes(8, 'big') + 
            ce.encode() + 
            ri_e + 
            self.hash_ciphertext(Ci)
        )
        return hashlib.sha256(data).digest()
    
    def sense(self):
        if self.last_message and random.random() < 0.3:  # Simulate 30% chance of retrying the last message
            print("Retrying last message due to simulated sensing error...")
            return self.last_message
        topics = ["temperature", "humidity", "heart_rate"]
        topic = random.choice(topics)
        ts = time.time()
        Ri = {
            "ID": self.device_id,
            "ts": ts,
            "ctr": self.counter,
            "topic": topic,
            "m": self.generate_payload(topic)
        }
    
        ts_bytes = struct.pack(">d", ts)          # float -> 8-byte IEEE 754
        device_id_bytes = self.device_id.encode() # str -> bytes

        Ki = HKDF(
            algorithm=hashes.SHA256(),
            length=32,
            salt=None,
            info=b'IoT data encryption',
        ).derive(device_id_bytes + ts_bytes + self.counter.to_bytes(8, 'big') + self.SS)

        # Generate nounce 
        nounce = os.urandom(12)

        self.counter += 1

        #Construct AEAD record (Ri, Ki, nounce)
        add_str = f"{Ri['ID']}|{Ri['ts']}|{Ri['ctr']}|{Ri['topic']}|{Ri['m']}"
        aad = add_str.encode()

        #Build protected record
        Ci = self.encrypt_record(Ki, nounce, Ri['m'], aad)
               
        #Generate authentication tag
        ri_e = self.simulate_puf(self.ds, ce=self.ce)
        Auth = self.generate_auth(Ri, ce=self.ce, ri_e=ri_e, Ci=Ci)

        final_message = {
            "ID": Ri['ID'],
            "ts": Ri['ts'],
            "ctr": Ri['ctr'],
            "topic": Ri['topic'],
            "nounce": nounce,
            "AAD": aad,
            "Ci": Ci,
            "ce": self.ce,
            "Auth": Auth
            }
        
        time.sleep(random.uniform(0, 1))  # Simulate sensing delay

        self.last_message = final_message  # Store the last message for potential retries
        return final_message

# Simulate 1 IoT device
device = IoTDevice("Device_1", hdfs_shared_secret_enc)

print("Simulating data acquisition from IoT device...")
for _ in range(n_records):
    final_message = device.sense()
    print("Final Message:", final_message)
    with open("iot_data_records.txt", "a") as f:
        f.write(str(final_message) + "\n")
    print()



Simulating data acquisition from IoT device...
Final Message: {'ID': 'Device_1', 'ts': 1778701308.496664, 'ctr': 0, 'topic': 'temperature', 'nounce': b'%Xo\xe8\xcf<\xd8\xfe\x85a\x9a\x94', 'AAD': b'Device_1|1778701308.496664|0|temperature|31.19', 'Ci': b'\x90\xf0\xce*\xdc\xe6.,\xdf0O\xf48\x94\x0cP\xba\xfa\xaab\x05', 'ce': 'epoch1', 'Auth': b')\xa3\xd8\r?\xbe\xc2\xb1\xd3\xf5\xf6\x96\xd7\x13\xa3\xc9\r\xa7\x87\xb6x4\xf3\xd7\x92\x0e\xf6"\x1f\x944\x10'}

Final Message: {'ID': 'Device_1', 'ts': 1778701309.0306623, 'ctr': 1, 'topic': 'heart_rate', 'nounce': b'[b)\x16l3\x85q\x0e\xd9MO', 'AAD': b'Device_1|1778701309.0306623|1|heart_rate|68', 'Ci': b'a\xc8\xfc6yc\xfd;\x16i\xf0\xebic\x876\xb7\xd8', 'ce': 'epoch1', 'Auth': b'\x05p\x01\xeas\xe3\x9f\x08O$z\x1e\x08\x9c\xb7\xae\xe2\xb0\xb6\xb1\xa9WLt\xad\xa7U\xe6\xe1\xeaD\x19'}

Final Message: {'ID': 'Device_1', 'ts': 1778701309.7957475, 'ctr': 2, 'topic': 'heart_rate', 'nounce': b'\\iU\x00\xd6`j\xedP\xfa\xeb6', 'AAD': b'Device_1|1778701309.7957475|2|h

Phase 3

In [10]:
#Simulate Kafka Stream Ingestion

class KafkaSimulator:
    def __init__(self, num_partitions):
        self.partitions = {i: [] for i in range(num_partitions)}
        self.num_partitions = num_partitions
    
    def partition_fn(self, ID, topic):
        # Simple hash-based partitioning
        return hash(ID + topic) % self.num_partitions
    
    def publish(self, Mi):
        p = self.partition_fn(Mi['ID'], Mi['topic'])
        self.partitions[p].append(Mi)
        print(f"Published message to partition {p}: {Mi}")

    def get_partition(self):
        return self.partitions

# Simulate Spark Stream Processing

class SparkSimulator:
    def __init__(self, delta):
        self.delta = delta
        self.epoch_batches = {}
        self.epoch_seen = {}
    
    def extract_metadata(self, Mi):
        metadata = {
            "ID": Mi['ID'],
            "ts": Mi['ts'],
            "ctr": Mi['ctr'],
            "topic": Mi['topic'],
            "ce": Mi['ce']
        }
        return metadata
    
    def process_stream(self, kafka_partitions):
        metadata_stream = {}
        for partition, messages in kafka_partitions.items():
            metadata_stream[partition] = []
            for Mi in messages:
                metadata = self.extract_metadata(Mi)
                metadata_stream[partition].append(metadata)
                print(f"Extracted metadata from partition {partition}: {metadata}")
        return metadata_stream
    
    def compute_epoch(self, ts):
        return int(ts // self.delta)
    
    def assign_epoch(self, Mi):
        e = self.compute_epoch(Mi['ts'])
        return e
    
    def form_batches(self, kafka_partitions):
        batches = {}
        for partition, messages in kafka_partitions.items():
            for Mi in messages:
                e = self.assign_epoch(Mi)
                batches.setdefault(e, []).append(Mi)

        for e, msgs in batches.items():
            batches[e] = self.deduplicate(msgs)

        return batches
    
    def compute_rid(self, Mi):
        data = (
            Mi['ID'].encode() +
            struct.pack(">d", Mi['ts']) + 
            Mi['ctr'].to_bytes(8, 'big') + 
            Mi['topic'].encode()
        )
        return hashlib.sha256(data).hexdigest()
    
    def deduplicate(self, batch):
        seen_rids = set()
        unique_batch = []
        for Mi in batch:
            rid = self.compute_rid(Mi)
            if rid not in seen_rids:
                seen_rids.add(rid)
                unique_batch.append(Mi)
            else:
                print(f"Duplicate detected and removed: {Mi}")
        return unique_batch
    
    def derive_context(self, batch):
        topics = set(msg["topic"] for msg in batch)
        if "heart_rate" in topics:
            return "healthcare"
        elif "temperature" in topics and "humidity" in topics:
            return "environment"
        else:
            return "generic"
    
    def map_policy(self, ctx):
        policy_map = {
            "healthcare": "POLICY_HEALTH_STRICT",
            "environment": "POLICY_ENV_MONITOR",
            "generic": "POLICY_DEFAULT"
        }
        return policy_map.get(ctx, "POLICY_DEFAULT")
    
    def assign_policy(self, epoch_batches):
        policy_results = {}
        for e,  batch in epoch_batches.items():
            ctx = self.derive_context(batch)
            policy = self.map_policy(ctx)
            policy_results[e] = {
                "context": ctx,
                "policy": policy,
                "batch": batch
            }
        return policy_results
    
    def hash_token(self, value):
        return hashlib.sha256(value.encode()).hexdigest()
    
    def generate_token(self, Mi, epoch, context):
        tokens = {
            "kw1": self.hash_token(Mi["ID"]),
            "kw2": self.hash_token(Mi["topic"]),
            "kw3": self.hash_token(str(epoch)),
            "kw4": self.hash_token(context + "|regionA")
        }
        return tokens
    
    def tokenize_batches(self, policy_batches):
        tokenized = {}
        for e, data in policy_batches.items():
            ctx = data["context"]
            batch = data["batch"]
            tokenized[e] = []
            for Mi in batch:
                tokens = self.generate_token(Mi, e, ctx)
                tokenized[e].append({
                    "Mi": Mi,
                    "tokens": tokens
                })
        return tokenized

    def compute_obj_id(self, Mi):
        hC = hashlib.sha256(Mi["Ci"]).digest()
        data = (
            Mi["ID"].encode() +
            struct.pack(">d", Mi["ts"]) +
            str(Mi["ctr"]).encode() +
            hC
        )
        return hashlib.sha256(data).hexdigest()
    
    def build_index_entry(self, tokens, obj_id):
        return {
            "tokens": tokens,
            "obj_id": obj_id
        }
    
    def build_epoch_index(self, tokenized_batches):
        epoch_indices = {}
        for e, records in tokenized_batches.items():
            epoch_indices[e] = []
            for r in records:
                Mi = r["Mi"]
                tokens = r["tokens"]
                obj_id = self.compute_obj_id(Mi)
                idx_entry = self.build_index_entry(tokens, obj_id)
                epoch_indices[e].append(idx_entry)
        return epoch_indices
    
    def compute_pre_digest(self, Mi, epoch):
        data = (
            Mi["ID"].encode() +
            struct.pack(">d", Mi["ts"]) +
            str(Mi["ctr"]).encode() +
            str(epoch).encode() +
            Mi["Ci"]
        )
        return hashlib.sha256(data).hexdigest()

    def compute_epoch_digests(self, epoch_batches):
        epoch_digests = {}
        for e, batch in epoch_batches.items():
            digests = []
            for Mi in batch:
                h_i = self.compute_pre_digest(Mi, e)
                digests.append(h_i)
            epoch_digests[e] = digests
        return epoch_digests
    
    def build_final_output(self, epoch_batches, epoch_indices, policy_batches, epoch_digests):
        final_output = {}
        for e in epoch_batches:
            De = epoch_batches[e]
            Ie = epoch_indices[e]
            H_e = epoch_digests[e]
            PolRef = policy_batches[e]["policy"]
            final_output[e] = {
                "De": De,
                "Ie": Ie,
                "H_e": H_e,
                "PolRef": PolRef,
                "epoch": e
            }
        return final_output

#Outputs

# Phase 3 Performance Monitoring
t_phase3_start = time.perf_counter()
t_kafka_start = time.perf_counter()

kafka_simulator = KafkaSimulator(num_partitions)
spark_simulator = SparkSimulator(delta)
with open("iot_data_records.txt", "r") as f:
    for line in f:
        final_message = eval(line)
        kafka_simulator.publish(final_message)

t_kafka_end = time.perf_counter()
t_spark_start = time.perf_counter()

print("Simulating Spark stream processing...")
metadata_stream = spark_simulator.process_stream(kafka_simulator.get_partition())
batches = spark_simulator.form_batches(kafka_simulator.get_partition())

t_spark_batch_end = time.perf_counter()
t_spark_policy_start = time.perf_counter()

policy_results = spark_simulator.assign_policy(batches)
tokenized_batches = spark_simulator.tokenize_batches(policy_results)
epoch_indices = spark_simulator.build_epoch_index(tokenized_batches)

t_spark_policy_end = time.perf_counter()
t_spark_output_start = time.perf_counter()

epoch_digests = spark_simulator.compute_epoch_digests(batches)
final_output = spark_simulator.build_final_output(batches, epoch_indices, policy_results, epoch_digests)
with open("Kafa-Spark_Output.txt", "w") as f:
    f.write(str(final_output) + "\n")

t_spark_output_end = time.perf_counter()
t_phase3_end = time.perf_counter()

# Store timing metrics
performance_metrics["phase_3"] = {
    "kafka_ingestion": t_kafka_end - t_kafka_start,
    "spark_batch_formation": t_spark_batch_end - t_spark_start,
    "spark_policy_tokenization": t_spark_policy_end - t_spark_policy_start,
    "spark_output_generation": t_spark_output_end - t_spark_output_start,
    "total_phase3": t_phase3_end - t_phase3_start,
    "total_messages": n_records,
    "num_epochs": len(final_output)
}

Published message to partition 5: {'ID': 'Device_1', 'ts': 1778701308.496664, 'ctr': 0, 'topic': 'temperature', 'nounce': b'%Xo\xe8\xcf<\xd8\xfe\x85a\x9a\x94', 'AAD': b'Device_1|1778701308.496664|0|temperature|31.19', 'Ci': b'\x90\xf0\xce*\xdc\xe6.,\xdf0O\xf48\x94\x0cP\xba\xfa\xaab\x05', 'ce': 'epoch1', 'Auth': b')\xa3\xd8\r?\xbe\xc2\xb1\xd3\xf5\xf6\x96\xd7\x13\xa3\xc9\r\xa7\x87\xb6x4\xf3\xd7\x92\x0e\xf6"\x1f\x944\x10'}
Published message to partition 4: {'ID': 'Device_1', 'ts': 1778701309.0306623, 'ctr': 1, 'topic': 'heart_rate', 'nounce': b'[b)\x16l3\x85q\x0e\xd9MO', 'AAD': b'Device_1|1778701309.0306623|1|heart_rate|68', 'Ci': b'a\xc8\xfc6yc\xfd;\x16i\xf0\xebic\x876\xb7\xd8', 'ce': 'epoch1', 'Auth': b'\x05p\x01\xeas\xe3\x9f\x08O$z\x1e\x08\x9c\xb7\xae\xe2\xb0\xb6\xb1\xa9WLt\xad\xa7U\xe6\xe1\xeaD\x19'}
Published message to partition 4: {'ID': 'Device_1', 'ts': 1778701309.7957475, 'ctr': 2, 'topic': 'heart_rate', 'nounce': b'\\iU\x00\xd6`j\xedP\xfa\xeb6', 'AAD': b'Device_1|1778701309.795